# Preprocess movie metadata

Clean and engineer features from `movies_metadata.csv` for clustering and prediction. Save the model-ready table to `data/processed/movies_features.csv`.

## 1. Load metadata

Read the joined Netflix–TMDB movie table.

In [11]:
from pathlib import Path
import ast
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

ROOT = Path("..")
movies = pd.read_csv(ROOT / "data/processed/movies_metadata.csv")
print(movies.shape)
movies.head(2)

(1806, 25)


,netflix_movie_id,tmdb_id,netflix_title,tmdb_title,year,budget,genres,homepage,keywords,original_language,...,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count,cast,crew
0,3,17139,Character,Character,1997.0,4500000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10769, ""n...",NaN,"[{""id"": 3683, ""name"": ""law and ethics""}]",nl,...,1997-04-17,0,122.0,"[{""iso_639_1"": ""nl"", ""name"": ""Nederlands""}, {""...",Released,NaN,7.7,25,"[{""cast_id"": 4, ""character"": ""Dreverhaven"", ""c...","[{""credit_id"": ""52fe47099251416c7508bca3"", ""de..."
1,30,6964,Something's Gotta Give,Something's Gotta Give,2003.0,80000000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 35, ""name...",NaN,"[{""id"": 965, ""name"": ""age difference""}, {""id"":...",en,...,2003-12-12,266728738,128.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,Schmucks are people too.,6.3,410,"[{""cast_id"": 1, ""character"": ""Harry Sanborn"", ...","[{""credit_id"": ""548200699251416e7800666d"", ""de..."


## 2. Drop unused columns and unify title

Remove columns that are sparse, constant, or redundant with ratings we will compute later. Keep a single title (`netflix_title`). Drop free-text `overview` (not used without NLP).

In [12]:
DROP_COLS = [
    "homepage",
    "tagline",
    "keywords",
    "status",
    "popularity",
    "vote_average",
    "vote_count",
    "tmdb_title",
    "original_title",
    "overview",
]

df = movies.drop(columns=DROP_COLS).rename(columns={"netflix_title": "title"})
print(df.shape)
print(list(df.columns))

(1806, 15)
['netflix_movie_id', 'tmdb_id', 'title', 'year', 'budget', 'genres', 'original_language', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'cast', 'crew']


## 3. Remove movies with zero budget, revenue, or runtime

TMDB encodes unknown budget/revenue/runtime as 0. Drop any row where budget, revenue, or runtime is zero so later numeric features use only observed production values.

In [13]:
n_before = len(df)
zero_mask = (df["budget"] == 0) | (df["revenue"] == 0) | (df["runtime"] == 0)
df = df[~zero_mask].copy()
print(f"removed {n_before - len(df)} rows with budget/revenue/runtime == 0; remaining {len(df)}")
print(
    "zeros left - budget:",
    int((df["budget"] == 0).sum()),
    "revenue:",
    int((df["revenue"] == 0).sum()),
    "runtime:",
    int((df["runtime"] == 0).sum()),
)

removed 484 rows with budget/revenue/runtime == 0; remaining 1322
zeros left - budget: 0 revenue: 0 runtime: 0


## 4. Parse JSON fields and extract cast/crew

Parse list-of-dict string columns into name lists. Extract the director and top-3 billed actors (feature slots). Keep the full cast name list for in-dataset film-count frequencies in section 6.


In [14]:
def parse_names(val):
    if pd.isna(val) or val == "":
        return []
    try:
        obj = ast.literal_eval(val) if isinstance(val, str) else val
    except (ValueError, SyntaxError):
        return []
    if not isinstance(obj, list):
        return []
    return [x.get("name") for x in obj if isinstance(x, dict) and x.get("name")]


def extract_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str) if isinstance(crew_str, str) else crew_str
    except (ValueError, SyntaxError):
        return None
    if not isinstance(crew, list):
        return None
    for person in crew:
        if isinstance(person, dict) and person.get("job") == "Director":
            return person.get("name")
    return None


def extract_top_actors(cast_str, n=3):
    try:
        cast = ast.literal_eval(cast_str) if isinstance(cast_str, str) else cast_str
    except (ValueError, SyntaxError):
        return [None] * n
    if not isinstance(cast, list):
        return [None] * n
    cast_sorted = sorted(
        [c for c in cast if isinstance(c, dict)],
        key=lambda c: c.get("order", 999),
    )
    names = [c.get("name") for c in cast_sorted[:n]]
    return names + [None] * (n - len(names))


df["genre_names"] = df["genres"].map(parse_names)
df["company_names"] = df["production_companies"].map(parse_names)
df["country_names"] = df["production_countries"].map(parse_names)
df["language_names"] = df["spoken_languages"].map(parse_names)
df["cast_names"] = df["cast"].map(parse_names)  # full cast for film counts
df["director"] = df["crew"].map(extract_director)

actors = df["cast"].map(lambda s: extract_top_actors(s, 3))
df["actor_1"] = actors.map(lambda x: x[0])
df["actor_2"] = actors.map(lambda x: x[1])
df["actor_3"] = actors.map(lambda x: x[2])

print("directors missing:", int(df["director"].isna().sum()))
print("cast name list lengths (sample):", df["cast_names"].map(len).describe()[["mean", "50%", "max"]].to_dict())
df[["title", "director", "actor_1", "actor_2", "actor_3"]].head(3)

directors missing: 0
cast name list lengths (sample): {'mean': 23.876701966717096, '50%': 18.0, 'max': 213.0}


,title,director,actor_1,actor_2,actor_3
1,Something's Gotta Give,Nancy Meyers,Jack Nicholson,Diane Keaton,Keanu Reeves
2,Jade,William Friedkin,David Caruso,Linda Fiorentino,Chazz Palminteri
4,Dragonheart,Rob Cohen,Dennis Quaid,David Thewlis,Pete Postlethwaite


## 5. Numeric features

Apply log1p to budget and revenue (revenue_log is only used to form log_roi), compute log_roi = revenue_log - budget_log, and add movie_age = 2026 - year. Keep budget_log + log_roi (drop revenue_log from the model columns); keep movie_age and drop year. Film-count logs and StandardScaler run in section 6 once name frequencies exist. No imputation is needed after dropping zero budget/revenue/runtime.


In [15]:
df["budget_log"] = np.log1p(df["budget"])
df["revenue_log"] = np.log1p(df["revenue"])  # intermediate for log_roi only
df["log_roi"] = df["revenue_log"] - df["budget_log"]
df["movie_age"] = 2026 - df["year"]

df[["budget_log", "runtime", "log_roi", "movie_age"]].describe()

,budget_log,runtime,log_roi,movie_age
count,1322.000000,1322.000000,1322.000000,1322.000000
mean,16.710602,112.732980,0.803121,32.196672
std,1.641761,21.765121,1.582214,12.897631
min,0.693147,69.000000,-14.386055,21.000000
25%,16.118096,97.000000,0.069602,24.000000
50%,17.034386,108.000000,0.848827,28.000000
75%,17.727534,124.000000,1.562155,35.000000
max,19.113828,219.000000,15.262430,110.000000


## 6. Categorical features

- Genres: multi-hot for all genres
- Decade: one-hot from release year (`year // 10 * 10`, e.g. 1987 -> 1980)
- Language: `lang_en`; spoken languages -> `is_multilingual`
- Countries: multi-hot for top 10 + `country_other`
- Companies: multi-hot for top 20 + `company_other`
- Directors / actors: film-count track-record features (`*_film_count` and `*_film_count_log`) instead of sparse name one-hots. Director count from `director`; actor counts from the **full cast** across movies, then mapped onto top-3 billing slots `actor_1`...`actor_3`.


In [16]:
def slugify(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


feature_blocks = []

# language
feature_blocks.append(
    pd.DataFrame(
        {
            "lang_en": (df["original_language"] == "en").astype(int),
            "is_multilingual": (df["language_names"].map(len) > 1).astype(int),
        },
        index=df.index,
    )
)

# decade from release year (e.g. 1987 -> 1980)
df["decade"] = (df["year"] // 10 * 10).astype(int)
print("decade counts:")
print(df["decade"].value_counts().sort_index())
decade_dummies = pd.get_dummies(df["decade"], prefix="decade").astype(int)
feature_blocks.append(decade_dummies)

# genres
all_genres = sorted({g for gs in df["genre_names"] for g in gs})
genre_cols = {
    f"genre_{slugify(g)}": df["genre_names"].map(lambda gs, g=g: int(g in gs))
    for g in all_genres
}
feature_blocks.append(pd.DataFrame(genre_cols, index=df.index))
print(f"genres: {len(all_genres)}")

# top 10 countries
country_counts = Counter(c for cs in df["country_names"] for c in cs)
top_countries = [c for c, _ in country_counts.most_common(10)]
country_cols = {
    f"country_{slugify(c)}": df["country_names"].map(lambda cs, c=c: int(c in cs))
    for c in top_countries
}
country_cols["country_other"] = df["country_names"].map(
    lambda cs: int(any(c not in top_countries for c in cs))
)
feature_blocks.append(pd.DataFrame(country_cols, index=df.index))
print("top countries:", top_countries)

# top 20 companies
company_counts = Counter(c for cs in df["company_names"] for c in cs)
top_companies = [c for c, _ in company_counts.most_common(20)]
company_cols = {
    f"company_{slugify(c)}": df["company_names"].map(lambda cs, c=c: int(c in cs))
    for c in top_companies
}
company_cols["company_other"] = df["company_names"].map(
    lambda cs: int(len(cs) == 0 or any(c not in top_companies for c in cs))
)
feature_blocks.append(pd.DataFrame(company_cols, index=df.index))
print("top companies:", top_companies[:5], "...")

# in-dataset film counts: director from billing; actors from full cast, mapped to top-3 slots
dir_counts = df["director"].dropna().value_counts()
actor_counts = pd.Series(
    [name for names in df["cast_names"] for name in names]
).value_counts()

df["director_film_count"] = df["director"].map(dir_counts).fillna(0)
df["director_film_count_log"] = np.log1p(df["director_film_count"])
for i in (1, 2, 3):
    df[f"actor_{i}_film_count"] = df[f"actor_{i}"].map(actor_counts).fillna(0)
    df[f"actor_{i}_film_count_log"] = np.log1p(df[f"actor_{i}_film_count"])

print(
    f"unique directors: {len(dir_counts)}; unique actors (full cast): {len(actor_counts)}"
)

# scale continuous model columns (film counts ready)
NUM_COLS = [
    "budget_log",
    "runtime",
    "log_roi",
    "director_film_count_log",
    "actor_1_film_count_log",
    "actor_2_film_count_log",
    "actor_3_film_count_log",
]
scaler = StandardScaler()
scaled = scaler.fit_transform(df[NUM_COLS])
scaled_df = pd.DataFrame(
    scaled,
    index=df.index,
    columns=[f"{c}_scaled" for c in NUM_COLS],
)
df = pd.concat([df, scaled_df], axis=1)

cat_features = pd.concat(feature_blocks, axis=1)
print("categorical feature columns:", cat_features.shape[1])


decade counts:
decade
1910      1
1920      1
1930      7
1940     13
1950     19
1960     50
1970     58
1980    149
1990    462
2000    562
Name: count, dtype: int64
genres: 19
top countries: ['United States of America', 'United Kingdom', 'Germany', 'Canada', 'France', 'Australia', 'Italy', 'Japan', 'Ireland', 'Spain']
top companies: ['Paramount Pictures', 'Warner Bros.', 'Universal Pictures', 'Twentieth Century Fox Film Corporation', 'Columbia Pictures Corporation'] ...
unique directors: 704; unique actors (full cast): 19563
categorical feature columns: 63


## 7. Assemble feature table and save

Keep IDs, title, engineered numeric columns (raw + scaled), and categorical encodings. Drop year / revenue_log from the export (movie_age and budget_log + log_roi remain). Drop raw JSON and intermediate name lists.


In [17]:
KEEP_BASE = [
    "netflix_movie_id",
    "tmdb_id",
    "title",
    "movie_age",
    "runtime",
    "budget_log",
    "log_roi",
    "director_film_count",
    "actor_1_film_count",
    "actor_2_film_count",
    "actor_3_film_count",
    "director_film_count_log",
    "actor_1_film_count_log",
    "actor_2_film_count_log",
    "actor_3_film_count_log",
    "budget_log_scaled",
    "runtime_scaled",
    "log_roi_scaled",
    "director_film_count_log_scaled",
    "actor_1_film_count_log_scaled",
    "actor_2_film_count_log_scaled",
    "actor_3_film_count_log_scaled",
]

features = pd.concat([df[KEEP_BASE], cat_features], axis=1)

print("shape:", features.shape)
print("null count:", int(features.isna().sum().sum()))
print("columns (first 25):", list(features.columns[:25]))
features.head(2)


shape: (1322, 85)
null count: 0
columns (first 25): ['netflix_movie_id', 'tmdb_id', 'title', 'movie_age', 'runtime', 'budget_log', 'log_roi', 'director_film_count', 'actor_1_film_count', 'actor_2_film_count', 'actor_3_film_count', 'director_film_count_log', 'actor_1_film_count_log', 'actor_2_film_count_log', 'actor_3_film_count_log', 'budget_log_scaled', 'runtime_scaled', 'log_roi_scaled', 'director_film_count_log_scaled', 'actor_1_film_count_log_scaled', 'actor_2_film_count_log_scaled', 'actor_3_film_count_log_scaled', 'lang_en', 'is_multilingual', 'decade_1910']


,netflix_movie_id,tmdb_id,title,movie_age,runtime,budget_log,log_roi,director_film_count,actor_1_film_count,actor_2_film_count,...,company_walt_disney_pictures,company_dreamworks_skg,company_village_roadshow_pictures,company_tristar_pictures,company_amblin_entertainment,company_regency_enterprises,company_dimension_films,company_morgan_creek_productions,company_epsilon_motion_pictures,company_other
1,30,6964,Something's Gotta Give,23.0,128.0,18.197537,1.204206,2,9,7.0,...,0,0,0,0,0,0,0,0,0,1
2,55,11863,Jade,31.0,95.0,17.727534,-1.624388,5,4,3.0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
out_path = ROOT / "data/processed/movies_features.csv"
features.to_csv(out_path, index=False)
print(f"saved {out_path} ({features.shape[0]} rows x {features.shape[1]} cols)")

saved ..\data\processed\movies_features.csv (1322 rows x 85 cols)
